# 🔴 Model 3 — Customer Churn Prediction
**Agent 3 · Supervised Machine Learning**

**Run each cell one by one using Shift+Enter**

---
| Step | What happens |
|------|--------------|
| Load | Read `data_rfm.csv` (output of EDA notebook) |
| Features | RFM + AvgDaysBetweenOrders + ReturnRate + Country + FavCategory |
| Label | `IsChurned` = 1 if Recency > 90 days, else 0 |
| Models | Random Forest (primary) · XGBoost (secondary) |
| Evaluate | Accuracy, F1, ROC-AUC, Confusion Matrix |
| Output | `data_churn_scored.csv` — churn probability per customer |

> **Why 90 days?**  
> Retail industry standard: a customer who hasn't bought in 3 months is considered at-risk/churned.  
> You can change `CHURN_DAYS = 90` below to tune the threshold.

## Cell 1 — Install & Import

In [ ]:
!pip install xgboost scikit-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)
from xgboost import XGBClassifier

plt.rcParams.update({
    "figure.figsize"   : (14, 5),
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "axes.grid"        : True,
    "font.size"        : 11,
})
C = ["#534AB7", "#1D9E75", "#D85A30", "#BA7517", "#185FA5", "#993C1D"]

# ── tuneable global constant ──────────────────────────────────────────────────
CHURN_DAYS = 90     # customers inactive > N days are labelled churned = 1
RANDOM_STATE = 42
# ─────────────────────────────────────────────────────────────────────────────

print("Setup done ✓")

## Cell 2 — Load RFM Data from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE = "/content/drive/MyDrive/retail_project/"
RFM_FILE = SAVE + "data_rfm.csv"

rfm = pd.read_csv(RFM_FILE)

# Re-apply churn label with the global constant (EDA used 90 days too)
rfm["IsChurned"] = (rfm["Recency"] > CHURN_DAYS).astype(int)

print(f"RFM loaded : {rfm.shape[0]:,} customers x {rfm.shape[1]} columns")
print(f"\nLabel distribution (CHURN_DAYS = {CHURN_DAYS}):")
vc = rfm["IsChurned"].value_counts()
print(f"  Active  (0) : {vc.get(0, 0):,}  ({vc.get(0,0)/len(rfm)*100:.1f}%)")
print(f"  Churned (1) : {vc.get(1, 0):,}  ({vc.get(1,0)/len(rfm)*100:.1f}%)")
rfm.head(3)

## Cell 3 — Feature Selection & Encoding

Features used:
- **RFM core** : Recency, Frequency, Monetary
- **Behavioural** : UniqueProducts, AvgOrderValue, TotalItems, AvgDaysBetweenOrders, CustomerLifetimeDays
- **Risk signals** : ReturnRate, ReturnCount
- **Categorical** : Country (top 10 + "Other"), FavCategory

> `Recency` is kept as a feature — the model learns the non-linear relationship between recency and churn probability.

In [ ]:
# ── Categorical encoding ──────────────────────────────────────────────────────
# Keep top-10 countries, bucket the rest as "Other" (avoids high-cardinality dummies)
top_countries = rfm["Country"].value_counts().head(10).index.tolist()
rfm["CountryGroup"] = rfm["Country"].where(rfm["Country"].isin(top_countries), other="Other")

le_country  = LabelEncoder()
le_category = LabelEncoder()
rfm["CountryEnc"]  = le_country.fit_transform(rfm["CountryGroup"])
rfm["CategoryEnc"] = le_category.fit_transform(rfm["FavCategory"].fillna("Other"))

# ── Feature matrix ────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "Recency",
    "Frequency",
    "Monetary",
    "UniqueProducts",
    "AvgOrderValue",
    "TotalItems",
    "AvgDaysBetweenOrders",
    "CustomerLifetimeDays",
    "ReturnRate",
    "ReturnCount",
    "CountryEnc",
    "CategoryEnc",
]

# Drop any columns not present (e.g. if ReturnRate wasn't added in EDA)
FEATURE_COLS = [c for c in FEATURE_COLS if c in rfm.columns]

X = rfm[FEATURE_COLS].fillna(0)
y = rfm["IsChurned"]

print(f"Feature matrix : {X.shape}")
print(f"Features used  : {FEATURE_COLS}")
print(f"\nClass balance  : {y.value_counts().to_dict()}")

## Cell 4 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = RANDOM_STATE,
    stratify     = y          # keeps class ratio equal in both splits
)

print(f"Train  : {len(X_train):,} rows  |  Churned: {y_train.sum():,}  Active: {(y_train==0).sum():,}")
print(f"Test   : {len(X_test):,}  rows  |  Churned: {y_test.sum():,}   Active: {(y_test==0).sum():,}")
print(f"\nTrain churn rate : {y_train.mean()*100:.1f}%")
print(f"Test  churn rate : {y_test.mean()*100:.1f}%")

## Cell 5 — Model A: Random Forest (Primary SML)

Random Forest is an **ensemble of decision trees** — each tree sees a random subset of rows and features.  
- Naturally handles non-linear relationships
- Robust to outliers (Monetary has extreme values)
- Built-in feature importance

In [ ]:
rf = RandomForestClassifier(
    n_estimators = 300,
    max_depth    = 12,
    min_samples_leaf = 5,
    class_weight = "balanced",   # handles class imbalance automatically
    random_state = RANDOM_STATE,
    n_jobs       = -1
)

rf.fit(X_train, y_train)
y_pred_rf   = rf.predict(X_test)
y_proba_rf  = rf.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf)
auc_rf  = roc_auc_score(y_test, y_proba_rf)

print("═" * 45)
print("  RANDOM FOREST — Test Set Results")
print("═" * 45)
print(f"  Accuracy  : {acc_rf*100:.2f}%")
print(f"  F1 Score  : {f1_rf:.4f}")
print(f"  ROC-AUC   : {auc_rf:.4f}")
print("═" * 45)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=["Active", "Churned"]))

## Cell 6 — Model B: XGBoost (Secondary SML)

XGBoost is **gradient boosted trees** — each tree corrects the errors of the previous one.  
- Usually beats Random Forest on tabular data
- Handles missing values natively
- `scale_pos_weight` balances imbalanced classes

In [ ]:
# scale_pos_weight = #negative / #positive  (compensates for imbalance)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = round(neg / pos, 2)
print(f"scale_pos_weight = {neg} / {pos} = {spw}")

xgb = XGBClassifier(
    n_estimators      = 400,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = spw,
    use_label_encoder = False,
    eval_metric       = "logloss",
    random_state      = RANDOM_STATE,
    n_jobs            = -1
)

xgb.fit(X_train, y_train)
y_pred_xgb  = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb  = f1_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_proba_xgb)

print("\n" + "═" * 45)
print("  XGBOOST — Test Set Results")
print("═" * 45)
print(f"  Accuracy  : {acc_xgb*100:.2f}%")
print(f"  F1 Score  : {f1_xgb:.4f}")
print(f"  ROC-AUC   : {auc_xgb:.4f}")
print("═" * 45)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=["Active", "Churned"]))

## Cell 7 — Model Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model"    : ["Random Forest", "XGBoost"],
    "Accuracy" : [round(acc_rf, 4),  round(acc_xgb, 4)],
    "F1 Score" : [round(f1_rf, 4),   round(f1_xgb, 4)],
    "ROC-AUC"  : [round(auc_rf, 4),  round(auc_xgb, 4)],
})

# Highlight the winner for each metric
def highlight_max(s):
    return ["background-color: #d4edda; font-weight: bold" if v == s.max() else "" for v in s]

print("Model Comparison — Test Set")
comparison.style.apply(highlight_max, subset=["Accuracy", "F1 Score", "ROC-AUC"])

## Cell 8 — Confusion Matrices (Side-by-Side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Confusion Matrices — Test Set", fontsize=13, fontweight="bold")

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_rf, y_pred_xgb],
    ["Random Forest", "XGBoost"],
    [C[0], C[2]]
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm,
        annot=True, fmt=",d",
        cmap=sns.light_palette(color, as_cmap=True),
        ax=ax,
        linewidths=0.5,
        xticklabels=["Active", "Churned"],
        yticklabels=["Active", "Churned"]
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()
print("\nKey: TN=top-left, FP=top-right, FN=bottom-left, TP=bottom-right")
print("For churn: high TP (correct churn catch) and low FN (missed churners) matters most")

## Cell 9 — ROC Curves & AUC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test, y_proba_rf,
    name=f"Random Forest  (AUC={auc_rf:.3f})",
    color=C[0], ax=ax
)
RocCurveDisplay.from_predictions(
    y_test, y_proba_xgb,
    name=f"XGBoost        (AUC={auc_xgb:.3f})",
    color=C[2], ax=ax
)

ax.plot([0, 1], [0, 1], "k--", label="Random guess (AUC=0.500)")
ax.set_title("ROC Curves — Churn Prediction", fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Cell 10 — Feature Importance: Random Forest vs XGBoost

In [ ]:
fi_rf = pd.Series(rf.feature_importances_,  index=FEATURE_COLS).sort_values(ascending=True)
fi_xgb = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Feature Importance Comparison", fontsize=13, fontweight="bold")

for ax, fi, title, color in zip(
    axes, [fi_rf, fi_xgb],
    ["Random Forest", "XGBoost"],
    [C[0], C[2]]
):
    bars = ax.barh(fi.index, fi.values, color=color, alpha=0.85, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Importance Score")
    # Annotate bar values
    for bar, val in zip(bars, fi.values):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop 3 features — Random Forest:", fi_rf.tail(3).index.tolist()[::-1])
print("Top 3 features — XGBoost      :", fi_xgb.tail(3).index.tolist()[::-1])

## Cell 11 — 5-Fold Cross Validation (Best Model)

Cross-validation confirms the model generalises — not just lucky on one test split.

In [ ]:
# Use whichever model had higher ROC-AUC
best_model = xgb if auc_xgb >= auc_rf else rf
best_name  = "XGBoost" if auc_xgb >= auc_rf else "Random Forest"
print(f"Best model: {best_name} (AUC={max(auc_xgb, auc_rf):.4f})\n")
print("Running 5-fold cross-validation... (may take ~30 seconds)")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_acc = cross_val_score(best_model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
cv_f1  = cross_val_score(best_model, X, y, cv=cv, scoring="f1",       n_jobs=-1)
cv_auc = cross_val_score(best_model, X, y, cv=cv, scoring="roc_auc",  n_jobs=-1)

print(f"\n{'Metric':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 44)
for name, scores in [("Accuracy", cv_acc), ("F1", cv_f1), ("ROC-AUC", cv_auc)]:
    print(f"{name:<12} {scores.mean():>8.4f} {scores.std():>8.4f} {scores.min():>8.4f} {scores.max():>8.4f}")

print(f"\n✓ Low std across folds = model generalises well, no overfitting")

## Cell 12 — Churn Probability Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Churn Probability Distribution — Test Set", fontsize=13, fontweight="bold")

for ax, proba, title, color in zip(
    axes,
    [y_proba_rf, y_proba_xgb],
    ["Random Forest", "XGBoost"],
    [C[0], C[2]]
):
    # Separate active vs churned
    active_proba  = proba[y_test == 0]
    churned_proba = proba[y_test == 1]

    ax.hist(active_proba,  bins=40, alpha=0.7, color=C[1], label="Active  (true 0)")
    ax.hist(churned_proba, bins=40, alpha=0.7, color=C[2], label="Churned (true 1)")
    ax.axvline(0.5, color="black", linestyle="--", label="Decision threshold = 0.5")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted Churn Probability")
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.show()
print("\nIdeal: two distinct peaks — active near 0, churned near 1")
print("Overlap in the middle = uncertain customers = target for retention campaigns")

## Cell 13 — Score All Customers (Full Dataset)

Now we score every customer — not just the test split.  
This is the output that flows back to **Agent 1 (Segmentation)** and the **Power BI dashboard**.

In [ ]:
# Score with best model (full dataset)
X_all = rfm[FEATURE_COLS].fillna(0)

rfm["ChurnProba_RF"]  = rf.predict_proba(X_all)[:, 1].round(4)
rfm["ChurnProba_XGB"] = xgb.predict_proba(X_all)[:, 1].round(4)

# Use best model as the primary score
rfm["ChurnProbability"] = rfm[f"ChurnProba_{'XGB' if auc_xgb >= auc_rf else 'RF'}"]

# Churn risk tier for Power BI
def risk_tier(p):
    if p >= 0.75: return "High Risk"
    if p >= 0.50: return "Medium Risk"
    if p >= 0.25: return "Low Risk"
    return "Safe"

rfm["ChurnRiskTier"] = rfm["ChurnProbability"].apply(risk_tier)

tier_counts = rfm["ChurnRiskTier"].value_counts()
print("Churn Risk Tier distribution (all customers):")
for tier in ["High Risk", "Medium Risk", "Low Risk", "Safe"]:
    n = tier_counts.get(tier, 0)
    print(f"  {tier:<14} : {n:,}  ({n/len(rfm)*100:.1f}%)")

rfm[["Customer ID", "Recency", "Frequency", "Monetary",
     "IsChurned", "ChurnProbability", "ChurnRiskTier"]].head(8)

## Cell 14 — Churn Risk Tier Visualisation

In [ ]:
tier_order  = ["High Risk", "Medium Risk", "Low Risk", "Safe"]
tier_colors = [C[2], C[3], C[1], C[0]]
tier_vals   = [tier_counts.get(t, 0) for t in tier_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Churn Risk Tier Summary", fontsize=13, fontweight="bold")

# Bar chart
axes[0].bar(tier_order, tier_vals, color=tier_colors, edgecolor="white")
for i, v in enumerate(tier_vals):
    axes[0].text(i, v + max(tier_vals)*0.01, f"{v:,}", ha="center", fontsize=10)
axes[0].set_title("Customer Count per Tier", fontweight="bold")
axes[0].set_ylabel("Customers")

# Pie chart
axes[1].pie(
    tier_vals,
    labels=[f"{t}\n({v/len(rfm)*100:.1f}%)" for t, v in zip(tier_order, tier_vals)],
    colors=tier_colors,
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
axes[1].set_title("Risk Tier Share", fontweight="bold")

plt.tight_layout()
plt.show()

## Cell 15 — Top 20 High-Risk Customers

These are the **churned customers with the highest monetary value** — highest-priority for win-back campaigns.

In [ ]:
top_atrisk = (
    rfm[rfm["ChurnRiskTier"] == "High Risk"]
    .sort_values("Monetary", ascending=False)
    .head(20)
    [["Customer ID", "Recency", "Frequency", "Monetary",
      "ChurnProbability", "ChurnRiskTier", "Country"]]
    .reset_index(drop=True)
)

print(f"Top 20 High-Risk customers by lifetime spend:")
print(f"(These customers haven't purchased in >{CHURN_DAYS} days but spent the most historically)\n")
top_atrisk.style \
    .background_gradient(subset=["ChurnProbability"], cmap="Reds") \
    .background_gradient(subset=["Monetary"],         cmap="Greens") \
    .format({"Monetary": "£{:,.0f}", "ChurnProbability": "{:.2%}"})

## Cell 16 — Recency vs Churn Probability Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

scatter = ax.scatter(
    rfm["Recency"],
    rfm["ChurnProbability"],
    c=rfm["IsChurned"],
    cmap="RdYlGn_r",
    alpha=0.4, s=10,
    edgecolors="none"
)

ax.axvline(CHURN_DAYS, color="black", linestyle="--", linewidth=1.2,
           label=f"Churn threshold ({CHURN_DAYS} days)")
ax.axhline(0.5, color="grey", linestyle=":", linewidth=1,
           label="Decision boundary (prob = 0.50)")

plt.colorbar(scatter, ax=ax, label="IsChurned (0=Active, 1=Churned)")
ax.set_title("Recency vs Churn Probability", fontweight="bold")
ax.set_xlabel("Recency (days since last purchase)")
ax.set_ylabel("Churn Probability")
ax.legend()
plt.tight_layout()
plt.show()
print("\nNote: customers to the RIGHT of the dashed line are labelled churned in the training data")
print("The model learns a probability — not a hard cut-off — which is more useful for CRM")

## Cell 17 — Save Output to Google Drive

In [ ]:
import os

# Build the final scored table
churn_output = rfm[[
    "Customer ID",
    "Country",
    "Recency",
    "Frequency",
    "Monetary",
    "AvgOrderValue",
    "AvgDaysBetweenOrders",
    "CustomerLifetimeDays",
    "ReturnRate",
    "FavCategory",
    "IsChurned",
    "ChurnProba_RF",
    "ChurnProba_XGB",
    "ChurnProbability",   # best model
    "ChurnRiskTier",
]].copy()

OUT_PATH = SAVE + "data_churn_scored.csv"
churn_output.to_csv(OUT_PATH, index=False)

print("Saved to Google Drive:")
print(f"  data_churn_scored.csv  {len(churn_output):,} rows  →  Agent 1 (Segmentation) + Power BI")
print(f"\nColumns saved: {list(churn_output.columns)}")
print(f"\nBest model used for ChurnProbability: {best_name}")
print(f"  RF  AUC = {auc_rf:.4f}")
print(f"  XGB AUC = {auc_xgb:.4f}")

## Cell 18 — Model 3 Summary

Run this cell for a clean print-out of everything that happened.

In [ ]:
print('''
+----------------------------------------------------------+
|      MODEL 3 — CUSTOMER CHURN PREDICTION SUMMARY        |
+----------------------------------------------------------+
''')
print(f"  Input file    : data_rfm.csv ({len(rfm):,} customers)")
print(f"  Churn label   : Recency > {CHURN_DAYS} days  → IsChurned = 1")
print(f"  Features      : {len(FEATURE_COLS)} — RFM, behavioural, return signals, country")
print()
print(f"  {'Model':<20} {'Accuracy':>10} {'F1 Score':>10} {'ROC-AUC':>10}")
print(f"  {'-'*52}")
print(f"  {'Random Forest':<20} {acc_rf:>10.4f} {f1_rf:>10.4f} {auc_rf:>10.4f}")
print(f"  {'XGBoost':<20} {acc_xgb:>10.4f} {f1_xgb:>10.4f} {auc_xgb:>10.4f}")
print()
print(f"  Best model    : {best_name} (ROC-AUC = {max(auc_rf, auc_xgb):.4f})")
print()
tier_order = ["High Risk", "Medium Risk", "Low Risk", "Safe"]
tier_counts = rfm["ChurnRiskTier"].value_counts()
for tier in tier_order:
    n = tier_counts.get(tier, 0)
    print(f"  {tier:<14} : {n:,}  ({n/len(rfm)*100:.1f}%)")
print()
print("  Output file   : data_churn_scored.csv")
print("  Feeds into    : Agent 1 (Segmentation enrichment) + Power BI dashboard")
print()
print("  Next notebook : 04_Market_Basket_Analysis.ipynb  (Agent 4 · Apriori)")
print()

## Cell 19 — Interactive Churn Tester

Three tabs:
- **Customer Lookup** — type any Customer ID from your dataset
- **Hypothetical** — sliders to predict on any made-up customer
- **Threshold Tuner** — drag to change the decision boundary and see Precision/Recall shift

> Run this cell **after** Cell 13 (scoring) has finished.

In [ ]:
## Cell 19 — Interactive Churn Prediction Tester

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Styling (matches Agent 7 dark theme) ─────────────────────────────────────
display(HTML("""
<style>
  .churn-panel {
    background: #1e1e2e;
    border-left: 4px solid #534AB7;
    border-radius: 6px;
    padding: 16px 20px;
    margin-bottom: 10px;
    font-family: monospace;
  }
  .churn-title {
    color: #534AB7;
    font-weight: bold;
    font-size: 14px;
    margin-bottom: 2px;
  }
  .churn-subtitle {
    color: #888;
    font-size: 11px;
    margin-bottom: 14px;
  }
  .section-label {
    color: #534AB7;
    font-weight: bold;
    font-size: 12px;
    margin: 10px 0 4px 0;
  }
  .output-box {
    background: #111;
    border: 1px solid #333;
    border-radius: 4px;
    padding: 14px;
    font-family: monospace;
    font-size: 12px;
    color: #ccc;
    white-space: pre;
    margin-top: 10px;
  }
</style>
"""))

# ── Shared style dicts ────────────────────────────────────────────────────────
_label_style  = {"description_width": "180px"}
_widget_layout = widgets.Layout(width="260px")
_text_layout   = widgets.Layout(width="260px", height="32px")

# ════════════════════════════════════════════════════════════════════════════
# TAB 1 — Customer ID Lookup
# ════════════════════════════════════════════════════════════════════════════
lbl_lookup = widgets.HTML("""
  <div class='churn-title'>Customer Churn Tester — Model 3</div>
  <div class='churn-subtitle'>Random Forest · XGBoost · RFM + Behavioural features</div>
  <div class='section-label'>🔍 Customer Lookup</div>
""")

cid_input = widgets.Text(
    value="",
    placeholder="e.g. 12345",
    description="Customer ID:",
    style=_label_style,
    layout=_text_layout
)

btn_lookup  = widgets.Button(
    description="Get Churn Profile",
    button_style="",
    layout=widgets.Layout(width="220px", height="36px"),
    style={"button_color": "#534AB7", "font_weight": "bold"}
)

out_lookup = widgets.Output()

def on_lookup(b):
    with out_lookup:
        clear_output(wait=True)
        cid = cid_input.value.strip()
        if not cid:
            print("⚠  Please enter a Customer ID.")
            return

        row = churn_output[churn_output["Customer ID"].astype(str) == cid]
        if row.empty:
            print(f"❌  Customer ID '{cid}' not found in data_churn_scored.")
            return

        r = row.iloc[0]

        # Risk colour
        tier = r["ChurnRiskTier"]
        tier_icon = {"High Risk": "🔴", "Medium Risk": "🟠", "Low Risk": "🟡", "Safe": "🟢"}.get(tier, "⚪")

        print("=" * 50)
        print("  CHURN PROFILE")
        print("=" * 50)
        print(f"  Customer ID          : {r['Customer ID']}")
        print(f"  Country              : {r['Country']}")
        print(f"  Favourite Category   : {r['FavCategory']}")
        print()
        print(f"  ── RFM ──────────────────────────────")
        print(f"  Recency              : {int(r['Recency'])} days since last purchase")
        print(f"  Frequency            : {int(r['Frequency'])} orders")
        print(f"  Monetary             : £{r['Monetary']:,.2f} lifetime spend")
        print(f"  Avg Order Value      : £{r['AvgOrderValue']:,.2f}")
        print(f"  Avg Days Btn Orders  : {r['AvgDaysBetweenOrders']} days")
        print(f"  Lifetime Days        : {int(r['CustomerLifetimeDays'])} days")
        print(f"  Return Rate          : {r['ReturnRate']*100:.1f}%")
        print()
        print(f"  ── CHURN SCORES ─────────────────────")
        print(f"  Random Forest Prob   : {r['ChurnProba_RF']*100:.1f}%")
        print(f"  XGBoost Prob         : {r['ChurnProba_XGB']*100:.1f}%")
        print(f"  Best Model Prob      : {r['ChurnProbability']*100:.1f}%")
        print(f"  True Label (>90d)    : {'Churned' if r['IsChurned'] else 'Active'}")
        print()
        print(f"  {tier_icon}  Risk Tier            : {tier}")
        print("=" * 50)

btn_lookup.on_click(on_lookup)

tab1_box = widgets.VBox([
    lbl_lookup,
    cid_input,
    btn_lookup,
    out_lookup
])

# ════════════════════════════════════════════════════════════════════════════
# TAB 2 — Hypothetical Customer (Sliders)
# ════════════════════════════════════════════════════════════════════════════
lbl_hypo = widgets.HTML("""
  <div class='section-label'>🧪 Hypothetical Customer</div>
  <div style='color:#888; font-size:11px; margin-bottom:8px;'>
    Adjust sliders → predict churn for any combination of features
  </div>
""")

sl_recency   = widgets.IntSlider(  value=60,  min=1,   max=365, step=1,  description="Recency (days):",       style=_label_style, layout=_widget_layout)
sl_frequency = widgets.IntSlider(  value=5,   min=1,   max=100, step=1,  description="Frequency (orders):",   style=_label_style, layout=_widget_layout)
sl_monetary  = widgets.FloatSlider(value=500, min=1,   max=20000, step=50, description="Monetary (£):",       style=_label_style, layout=_widget_layout)
sl_aov       = widgets.FloatSlider(value=80,  min=1,   max=2000,  step=5,  description="Avg Order Val (£):",  style=_label_style, layout=_widget_layout)
sl_days_btn  = widgets.FloatSlider(value=30,  min=0,   max=200,   step=1,  description="Avg Days Btn Orders:",style=_label_style, layout=_widget_layout)
sl_lifetime  = widgets.IntSlider(  value=200, min=0,   max=730,   step=5,  description="Lifetime Days:",      style=_label_style, layout=_widget_layout)
sl_items     = widgets.IntSlider(  value=50,  min=1,   max=5000,  step=10, description="Total Items:",        style=_label_style, layout=_widget_layout)
sl_unique    = widgets.IntSlider(  value=20,  min=1,   max=500,   step=1,  description="Unique Products:",    style=_label_style, layout=_widget_layout)
sl_ret_rate  = widgets.FloatSlider(value=0.0, min=0.0, max=1.0,   step=0.01, description="Return Rate:",     style=_label_style, layout=_widget_layout)
sl_ret_count = widgets.IntSlider(  value=0,   min=0,   max=50,    step=1,  description="Return Count:",       style=_label_style, layout=_widget_layout)

dd_country   = widgets.Dropdown(
    options=sorted(rfm["CountryGroup"].unique().tolist()),
    value="United Kingdom",
    description="Country:",
    style=_label_style, layout=_widget_layout
)
dd_category  = widgets.Dropdown(
    options=sorted(rfm["FavCategory"].fillna("Other").unique().tolist()),
    value="Other",
    description="Fav Category:",
    style=_label_style, layout=_widget_layout
)

btn_predict = widgets.Button(
    description="Predict Churn",
    layout=widgets.Layout(width="220px", height="36px"),
    style={"button_color": "#1D9E75", "font_weight": "bold"}
)

out_hypo = widgets.Output()

def on_predict(b):
    with out_hypo:
        clear_output(wait=True)

        country_enc  = le_country.transform([sl_country_val()])[0]  if sl_country_val() in le_country.classes_ else 0
        category_enc = le_category.transform([dd_category.value])[0] if dd_category.value in le_category.classes_ else 0

        row_dict = {
            "Recency"              : sl_recency.value,
            "Frequency"            : sl_frequency.value,
            "Monetary"             : sl_monetary.value,
            "UniqueProducts"       : sl_unique.value,
            "AvgOrderValue"        : sl_aov.value,
            "TotalItems"           : sl_items.value,
            "AvgDaysBetweenOrders" : sl_days_btn.value,
            "CustomerLifetimeDays" : sl_lifetime.value,
            "ReturnRate"           : sl_ret_rate.value,
            "ReturnCount"          : sl_ret_count.value,
            "CountryEnc"           : country_enc,
            "CategoryEnc"          : category_enc,
        }

        X_hypo = pd.DataFrame([{c: row_dict.get(c, 0) for c in FEATURE_COLS}])

        p_rf  = rf.predict_proba(X_hypo)[0][1]
        p_xgb = xgb.predict_proba(X_hypo)[0][1]
        p_best = p_xgb if auc_xgb >= auc_rf else p_rf

        tier = ("High Risk" if p_best >= 0.75 else
                "Medium Risk" if p_best >= 0.50 else
                "Low Risk"   if p_best >= 0.25 else "Safe")
        tier_icon = {"High Risk": "🔴", "Medium Risk": "🟠", "Low Risk": "🟡", "Safe": "🟢"}[tier]

        print("=" * 50)
        print("  HYPOTHETICAL CUSTOMER — CHURN PREDICTION")
        print("=" * 50)
        print(f"  Recency              : {sl_recency.value} days")
        print(f"  Frequency            : {sl_frequency.value} orders")
        print(f"  Monetary             : £{sl_monetary.value:,.0f}")
        print(f"  Avg Order Value      : £{sl_aov.value:,.0f}")
        print(f"  Avg Days Btn Orders  : {sl_days_btn.value} days")
        print(f"  Return Rate          : {sl_ret_rate.value*100:.0f}%")
        print()
        print(f"  Random Forest Prob   : {p_rf*100:.1f}%")
        print(f"  XGBoost Prob         : {p_xgb*100:.1f}%")
        print(f"  Best Model Prob      : {p_best*100:.1f}%")
        print()
        print(f"  {tier_icon}  Risk Tier            : {tier}")
        print("=" * 50)

def sl_country_val():
    return dd_country.value

btn_predict.on_click(on_predict)

tab2_box = widgets.VBox([
    lbl_hypo,
    sl_recency, sl_frequency, sl_monetary,
    sl_aov, sl_days_btn, sl_lifetime,
    sl_items, sl_unique,
    sl_ret_rate, sl_ret_count,
    dd_country, dd_category,
    btn_predict,
    out_hypo
])

# ════════════════════════════════════════════════════════════════════════════
# TAB 3 — Threshold Tuner
# ════════════════════════════════════════════════════════════════════════════
lbl_thresh = widgets.HTML("""
  <div class='section-label'>⚙️ Decision Threshold Tuner</div>
  <div style='color:#888; font-size:11px; margin-bottom:8px;'>
    Drag to change the classification boundary (default = 0.50).<br>
    Lower threshold → catch more churners (higher recall, more false positives).<br>
    Higher threshold → fewer false alarms (higher precision, miss some churners).
  </div>
""")

sl_threshold = widgets.FloatSlider(
    value=0.5, min=0.1, max=0.9, step=0.01,
    description="Threshold:",
    style=_label_style,
    layout=widgets.Layout(width="400px"),
    readout_format=".2f"
)

btn_tune = widgets.Button(
    description="Apply Threshold",
    layout=widgets.Layout(width="220px", height="36px"),
    style={"button_color": "#D85A30", "font_weight": "bold"}
)

out_thresh = widgets.Output()

def on_tune(b):
    with out_thresh:
        clear_output(wait=True)
        thresh = sl_threshold.value
        y_pred_tuned = (y_proba_xgb if auc_xgb >= auc_rf else y_proba_rf) >= thresh

        acc_t = accuracy_score(y_test, y_pred_tuned)
        f1_t  = f1_score(y_test, y_pred_tuned, zero_division=0)
        from sklearn.metrics import precision_score, recall_score
        prec_t = precision_score(y_test, y_pred_tuned, zero_division=0)
        rec_t  = recall_score(y_test, y_pred_tuned, zero_division=0)

        n_flagged = y_pred_tuned.sum()
        n_total   = len(y_pred_tuned)

        print("=" * 50)
        print(f"  THRESHOLD = {thresh:.2f}  ({best_name})")
        print("=" * 50)
        print(f"  Customers flagged as Churned : {n_flagged:,} / {n_total:,}  ({n_flagged/n_total*100:.1f}%)")
        print()
        print(f"  Accuracy   : {acc_t*100:.2f}%")
        print(f"  Precision  : {prec_t*100:.2f}%  (of flagged, how many truly churned)")
        print(f"  Recall     : {rec_t*100:.2f}%  (of all churners, how many we caught)")
        print(f"  F1 Score   : {f1_t:.4f}")
        print()
        if thresh < 0.4:
            print("  ⚠  Low threshold — high recall, many false alarms")
        elif thresh > 0.7:
            print("  ⚠  High threshold — very precise but misses many churners")
        else:
            print("  ✓  Balanced threshold — good for general CRM use")
        print("=" * 50)

btn_tune.on_click(on_tune)

tab3_box = widgets.VBox([
    lbl_thresh,
    sl_threshold,
    btn_tune,
    out_thresh
])

# ════════════════════════════════════════════════════════════════════════════
# Assemble Tabs
# ════════════════════════════════════════════════════════════════════════════
tabs = widgets.Tab(children=[tab1_box, tab2_box, tab3_box])
tabs.set_title(0, "🔍 Customer Lookup")
tabs.set_title(1, "🧪 Hypothetical")
tabs.set_title(2, "⚙️ Threshold Tuner")

display(tabs)
print("Tester ready — select a tab and fill inputs")
